In [ ]:
!pip install transformers datasets seqeval evaluate accelerate -q

In [ ]:
!pip install transformers datasets evaluate seqeval

In [ ]:
import numpy as np
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForTokenClassification, TrainingArguments, Trainer
from evaluate import load

In [ ]:
!pip install -U datasets


In [ ]:
from datasets import load_dataset

dataset = load_dataset("tner/conll2003")
print(dataset)

In [ ]:
!pip install -U datasets pyarrow

In [ ]:
!pip uninstall datasets -y
!pip install datasets==2.19.1

In [ ]:
from datasets import load_dataset

dataset = load_dataset("wnut_17")
print(dataset)

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True
    )

    labels = []

    for i, label in enumerate(examples["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []

        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])
            else:
                label_ids.append(-100)

            previous_word_idx = word_idx

        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

tokenized_dataset = dataset.map(tokenize_and_align_labels, batched=True)

In [ ]:
from transformers import AutoModelForTokenClassification

label_list = dataset["train"].features["ner_tags"].feature.names

model = AutoModelForTokenClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=len(label_list)
)

In [ ]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    num_train_epochs=2,
    weight_decay=0.01
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    data_collator=data_collator
)

In [ ]:
from transformers import DataCollatorForTokenClassification

data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

In [ ]:
trainer.train()

In [ ]:
trainer.evaluate()

In [ ]:
from evaluate import load
import numpy as np

metric = load("seqeval")

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_predictions = [
        [str(p) for (p, l) in zip(pred, lab) if l != -100]
        for pred, lab in zip(predictions, labels)
    ]

    true_labels = [
        [str(l) for (p, l) in zip(pred, lab) if l != -100]
        for pred, lab in zip(predictions, labels)
    ]

    results = metric.compute(predictions=true_predictions, references=true_labels)

    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
    }

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

In [ ]:
trainer.evaluate()

In [ ]:
The model achieved a precision of 0.36, recall of 0.16, and an F1-score of 0.22.
The relatively lower F1-score is due to the small dataset size and limited training epochs.
However, the model demonstrates the ability of transformer-based architectures to learn contextual relationships in sequence labeling tasks.

In [ ]:
In this project, I implemented a token classification model using BERT to perform named entity recognition. The dataset used was WNUT-17 from Hugging Face.

The preprocessing step involved tokenization and aligning labels with subword tokens. Special tokens were handled using -100 to ignore them during loss computation.

The model was trained using the Hugging Face Trainer API, and padding was handled using a data collator to manage variable-length sequences.

The model achieved a precision of 0.36, recall of 0.16, and F1-score of 0.22. Although the performance is moderate, it demonstrates the effectiveness of transformer models in sequence labeling tasks.

In [ ]:
| Feature    | POS Tagging     | Chunking         |
| ---------- | --------------- | ---------------- |
| Level      | Word level      | Phrase level     |
| Example    | NN, VB          | NP, VP           |
| Purpose    | Grammar tagging | Phrase detection |
| Difficulty | Easy            | Medium           |
